In [ ]:
from os import environ
from openai import OpenAI
import numpy as np
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer
import faiss

# Initialize OpenAI API - replace with your API key or use environment variable
openai_client = OpenAI(api_key=environ['OPENAI_API_KEY'])

# Initialize the embedding model
embedding_model = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
class RAGFusion:
    def __init__(self, documents: List[str], k: int = 60, model: str = "openai.gpt-4o"):
        """
        Initialize RAG-Fusion with a list of documents, RRF constant k, and OpenAI model.

        Args:
            documents: List of document texts
            k: Constant for Reciprocal Rank Fusion (default: 60)
            model: OpenAI model to use for generating queries and responses (default: "gpt-4o")
        """
        self.documents = documents
        self.k = k
        self.model = model

        # Create document embeddings
        self.document_embeddings = embedding_model.encode(documents)

        # Create FAISS index for fast vector search
        self.dimension = self.document_embeddings.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(np.array(self.document_embeddings).astype('float32'))

    def generate_queries(self, original_query: str, num_queries: int = 4) -> List[str]:
        """
        Generate multiple query variations from the original query using OpenAI.

        Args:
            original_query: The user's original query
            num_queries: Number of query variations to generate

        Returns:
            List of generated queries including the original query
        """
        try:
            response = openai_client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that generates multiple search queries based on a single input query."},
                    {"role": "user", "content": f"Generate {num_queries} different search queries related to: {original_query}"},
                    {"role": "user", "content": f"Make each query explore a different aspect or perspective of the original query. OUTPUT FORMAT: Return only the queries, one per line, with no numbering or additional text."}
                ]
            )

            # Extract and clean generated queries
            generated_text = response.choices[0].message.content.strip()
            generated_queries = [q.strip() for q in generated_text.split('\n') if q.strip()]

            # Add original query if not already included
            if original_query not in generated_queries:
                generated_queries.insert(0, original_query)

            return generated_queries

        except Exception as e:
            print(f"Error generating queries: {e}")
            # Fallback to just the original query
            return [original_query]

    def vector_search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """
        Perform vector search for a query.

        Args:
            query: The query text
            top_k: Number of top results to return

        Returns:
            List of (document_index, score) tuples
        """
        # Encode the query
        query_embedding = embedding_model.encode([query])[0].reshape(1, -1).astype('float32')

        # Search the index
        distances, indices = self.index.search(query_embedding, top_k)

        # Convert to list of (index, score) tuples
        # Note: Converting distance to similarity score (1 - normalized_distance)
        max_dist = np.max(distances)
        if max_dist > 0:
            normalized_distances = distances[0] / max_dist
        else:
            normalized_distances = distances[0]

        similarity_scores = 1 - normalized_distances
        results = [(int(indices[0][i]), float(similarity_scores[i]))
                  for i in range(len(indices[0]))]

        return results

    def reciprocal_rank_fusion(self, search_results: Dict[str, List[Tuple[int, float]]]) -> List[int]:
        """
        Apply Reciprocal Rank Fusion to combine results from multiple queries.

        Args:
            search_results: Dictionary mapping queries to their search results

        Returns:
            List of document indices, ranked by RRF score
        """
        fused_scores = {}

        # Print initial search results for demonstration
        print("\nInitial search results by query:")
        for query, results in search_results.items():
            print(f"Query: '{query}'")
            for rank, (doc_idx, score) in enumerate(results):
                doc_preview = self.documents[doc_idx][:50] + "..." if len(self.documents[doc_idx]) > 50 else self.documents[doc_idx]
                print(f"  Rank {rank+1}: Doc #{doc_idx} (Score: {score:.4f}) - {doc_preview}")

        # Calculate RRF scores
        for query, results in search_results.items():
            for rank, (doc_idx, _) in enumerate(results):
                if doc_idx not in fused_scores:
                    fused_scores[doc_idx] = 0

                # Apply RRF formula: 1 / (rank + k)
                rrf_score = 1.0 / (rank + self.k)
                fused_scores[doc_idx] += rrf_score

                print(f"Adding score for Doc #{doc_idx}: 1/({rank}+{self.k}) = {rrf_score:.6f}")

        # Sort documents by their fused scores in descending order
        ranked_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

        # Print final rankings
        print("\nFinal rankings after RRF:")
        for rank, (doc_idx, score) in enumerate(ranked_docs):
            doc_preview = self.documents[doc_idx][:50] + "..." if len(self.documents[doc_idx]) > 50 else self.documents[doc_idx]
            print(f"Rank {rank+1}: Doc #{doc_idx} (RRF Score: {score:.6f}) - {doc_preview}")

        return [doc_idx for doc_idx, _ in ranked_docs]

    def generate_response(self, original_query: str, ranked_docs: List[int], generated_queries: List[str]) -> str:
        """
        Generate a final response using the reranked documents and all queries.

        Args:
            original_query: The user's original query
            ranked_docs: List of document indices, ranked by relevance
            generated_queries: List of all queries used (original + generated)

        Returns:
            Generated response text
        """
        # Get the top documents
        top_docs = [self.documents[idx] for idx in ranked_docs[:3]]

        # Combine documents into context
        context = "\n\n".join([f"Document {i+1}:\n{doc}" for i, doc in enumerate(top_docs)])

        # Generate response using OpenAI
        try:
            response = openai_client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided documents."},
                    {"role": "user", "content": f"Original question: {original_query}\n\nI also explored these related questions:\n" +
                                               "\n".join([f"- {q}" for q in generated_queries if q != original_query]) +
                                               f"\n\nRelevant documents:\n{context}\n\nBased on these documents, please provide a comprehensive answer to my original question."}
                ]
            )

            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Error generating response: {e}")
            return f"I couldn't generate a response due to an error: {str(e)}"

    def process_query(self, query: str, num_generated_queries: int = 3, top_k: int = 5) -> str:
        """
        Process a query using RAG-Fusion and return a response.

        Args:
            query: The user's query
            num_generated_queries: Number of additional queries to generate
            top_k: Number of top results to retrieve per query

        Returns:
            Generated response
        """
        print(f"\n\n{'='*50}")
        print(f"Processing query: '{query}'")
        print(f"{'='*50}")

        # Step 1: Generate multiple queries
        print("\nGenerating multiple queries...")
        generated_queries = self.generate_queries(query, num_generated_queries)
        print(f"Generated queries:")
        for i, q in enumerate(generated_queries):
            print(f"{i+1}. {q}")

        # Step 2: Perform vector search for each query
        print("\nPerforming vector search for each query...")
        search_results = {}
        for q in generated_queries:
            search_results[q] = self.vector_search(q, top_k)

        # Step 3: Apply Reciprocal Rank Fusion
        print("\nApplying Reciprocal Rank Fusion...")
        ranked_docs = self.reciprocal_rank_fusion(search_results)

        # Step 4: Generate final response
        print("\nGenerating final response...")
        response = self.generate_response(query, ranked_docs, generated_queries)

        return response


In [ ]:
    # Sample documents about climate change
documents = [
    "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil, and gas, which produces heat-trapping gases.",
    "Global warming is the long-term heating of Earth's surface observed since the pre-industrial period due to human activities, primarily fossil fuel burning, which increases heat-trapping greenhouse gas levels in Earth's atmosphere.",
    "The effects of climate change include rising sea levels, increased frequency and severity of extreme weather events, shifts in plant blooming times, and wildlife population movements.",
    "Economic impacts of climate change include reduced agricultural productivity, increased health costs, property damage from floods and storms, and value loss in industries dependent on stable climate conditions.",
    "Climate change mitigation involves reducing the flow of heat-trapping greenhouse gases into the atmosphere, either by reducing sources of these gases or enhancing the sinks that accumulate and store these gases.",
    "Climate change adaptation refers to adjustments in ecological, social, or economic systems in response to actual or expected climatic stimuli and their effects or impacts.",
    "The Paris Agreement is a legally binding international treaty on climate change. It was adopted by 196 Parties at COP 21 in Paris on 12 December 2015 and entered into force on 4 November 2016.",
    "Renewable energy sources like solar, wind, and hydroelectric power are crucial for reducing greenhouse gas emissions and mitigating climate change.",
    "Carbon pricing is a method for reducing global warming emissions by charging emitters for the carbon they release. The two main types are carbon taxes and cap-and-trade programs.",
    "Climate justice is a term used to frame global warming as an ethical and political issue, rather than one that is purely environmental or physical in nature."
]

# Initialize RAG-Fusion
rag_fusion = RAGFusion(documents)

In [ ]:
# Process a query
query = "What are the economic impacts of climate change?"
response = rag_fusion.process_query(query)

print("\n\nFinal Response:")
print(f"{'-'*50}")
print(response)
print(f"{'-'*50}")

In [ ]:
    # Process another query to demonstrate versatility
query2 = "How can we address climate change?"
response2 = rag_fusion.process_query(query2)

print("\n\nFinal Response:")
print(f"{'-'*50}")
print(response2)
print(f"{'-'*50}")

